**Overview:** This notebook builds a complete, reproducible multi-label PyTorch baseline for predicting **12 knee pathology targets** from 3D MRI volumes. We utilize a 2.5D ResNet-34 architecture adapted for 24 spatial depth channels, training with Automatic Mixed Precision (AMP) on Fold 0 validation split.

## Datasets
- **Metadata & Pseudo-Labels CSV:** [RSNA Knee Metadata & Pseudo-Labels](https://www.kaggle.com/datasets/barun2104/rsna-knee-stratified-folds-and-llm-soft-labels)
- **Preprocessed 3D Volumes:** [RSNA Knee Processed 3D Volumes](https://www.kaggle.com/datasets/barun2104/rsna-knee-mri-processed-3d-volumes)

## 1. Environment Setup & Data Verification
Let's verify GPU availability and confirm that our preprocessed `(24, 224, 224)` `.npz` volume files are properly mounted in `/kaggle/input/`.

In [ ]:
import os
import time
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.metrics import roc_auc_score

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Using compute device: {device}")

# Path Specifications
IMG_DIR = "/kaggle/input/datasets/barun2104/rsna-knee-mri-processed-3d-volumes"
CSV_PATH = "/kaggle/input/datasets/barun2104/rsna-knee-stratified-folds-and-llm-soft-labels/train_folds_with_pseudo.csv"
TRAIN_SERIES_PATH = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv"

# Verify File Count in Input Directory
available_files = len([f for f in os.listdir(IMG_DIR) if f.endswith('.npz')])
print(f"✅ Input directory mounted: {available_files:,} volume files (.npz) detected.")

## 2. Load Metadata & Generate Study-Series Mapping 
Now, let's load our cross-validation metadata file containing pre-assigned stratified folds and target annotations across **12 primary knee pathologies.** We will use the `train_series.csv` to generate mapping between Study and Series.

In [ ]:
# 12 Multi-Label Target Pathologies
TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

# Load Custom Metadata
df = pd.read_csv(CSV_PATH)

# Load Competition train_series.csv
train_series_df = pd.read_csv(TRAIN_SERIES_PATH)

# Direct Mapping: StudyInstanceUID -> List of SeriesInstanceUIDs
study_to_series_map = train_series_df.groupby('StudyInstanceUID')['SeriesInstanceUID'].apply(list).to_dict()

# Attach series list directly to main dataframe
df['series_ids'] = df['StudyInstanceUID'].map(study_to_series_map)

# Filter for manually annotated ground-truth studies (fold != -1)
manual_df = df[df['fold'] != -1].reset_index(drop=True)

print(f"📊 Total Expert Ground-Truth Studies: {len(manual_df)}")
print(f"🔗 Total Mapped Studies in Series Mapping: {len(study_to_series_map):,}")
print(f"🎯 Target Count: {len(TARGET_COLS)} classes")
print("\nFold Distribution:")
print(manual_df['fold'].value_counts().sort_index())


## 3. PyTorch Dataset & DataLoader
Our dataset loader fetches each preprocessed `(24, 224, 224)` volume, normalizes pixel values from `uint8` (0-255) to `float32` (0.0-1.0), and returns the 12-element multi-label target vector.

In [ ]:
class KneeMRIDataset(Dataset):
    def __init__(self, df, series_map, img_dir=IMG_DIR, target_slices=24, is_train=True):
        self.df = df.reset_index(drop=True)
        self.series_map = series_map
        self.img_dir = img_dir
        self.target_slices = target_slices
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row['StudyInstanceUID']
        
        # 1. Fetch mapped SeriesInstanceUIDs for this study
        series_ids = self.series_map.get(study_id, [])
        
        # 2. Load all existing series .npz files for this study
        loaded_vols = []
        for s_id in series_ids:
            series_path = os.path.join(self.img_dir, f"{s_id}.npz")
            if os.path.exists(series_path):
                with np.load(series_path) as npz:
                    key = 'data' if 'data' in npz else npz.files[0]
                    vol = npz[key]  # Expected shape: (slices, H, W)
                    loaded_vols.append(vol)

        # 3. Raise explicit error if no series files exist
        if len(loaded_vols) == 0:
            raise FileNotFoundError(
                f"No series .npz files found for StudyInstanceUID '{study_id}' in directory: {self.img_dir}"
            )

        # 4. Concatenate loaded series along the depth (slice) axis
        combined_vol = np.concatenate(loaded_vols, axis=0)  # Shape: (total_slices, H, W)
        
        # 5. Resample slice count to fixed target_slices (24) via uniform indexing
        curr_slices = combined_vol.shape[0]
        if curr_slices != self.target_slices:
            slice_indices = np.linspace(0, curr_slices - 1, self.target_slices).astype(int)
            combined_vol = combined_vol[slice_indices]

        # 6. Convert to FloatTensor and scale uint8 [0, 255] -> float [0.0, 1.0]
        tensor_vol = torch.from_numpy(combined_vol).float()
        if tensor_vol.max() > 1.0:
            tensor_vol = tensor_vol / 255.0

        labels = torch.tensor(row[TARGET_COLS].values.astype(np.float32))
        return tensor_vol, labels
        
# Reserve Fold 0 for Validation, Folds 1-4 for Training
train_df = manual_df[manual_df['fold'] != 0].reset_index(drop=True)
val_df = manual_df[manual_df['fold'] == 0].reset_index(drop=True)

train_dataset = KneeMRIDataset(train_df, series_map=study_to_series_map, is_train=True)
val_dataset = KneeMRIDataset(val_df, series_map=study_to_series_map, is_train=False)

# DataLoaders (Batch size = 32 due to reduced memory footprint)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"🏋️ Train samples (Folds 1-4): {len(train_dataset)}")
print(f"🧪 Validation samples (Fold 0): {len(val_dataset)}")

## 4. 2.5D ResNet-34 Architecture
Instead of computationally expensive 3D convolutions, we adopt a **2.5D approach:**
- We treat the **24 depth slices** as input feature channels into a standard 2D Convolutional Backbone.
- The first layer (`conv1`) is adapted from 3 channels to 24 channels by repeating ResNet pre-trained weights across the depth axis, maintaining rich transfer-learning initialization.

In [ ]:
class KneeResNet25D(nn.Module):
    def __init__(self, num_classes=12, in_channels=24, pretrained=True):
        super().__init__()
        weights = models.ResNet34_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet34(weights=weights)
        
        # Modify conv1 for 24 depth slices
        old_conv = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(
            in_channels, old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias
        )
        
        # Proper weight expansion initialization
        with torch.no_grad():
            repeat_factor = (in_channels // 3) + 1
            new_weight = old_conv.weight.repeat(1, repeat_factor, 1, 1)[:, :in_channels, :, :]
            # Scale by sqrt ratio to maintain variance
            self.backbone.conv1.weight = nn.Parameter(new_weight * np.sqrt(3.0 / in_channels))
            
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

model = KneeResNet25D(num_classes=len(TARGET_COLS), in_channels=24, pretrained=True).to(device)
print(f"✅ Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters.")

## 5. Metric Evaluation & Loss Setup
We evaluate validation performance using **Macro ROC-AUC** across all 12 targets, handling edge cases where a batch or split might have only single-class instances.

In [ ]:
def compute_macro_auc(y_true, y_pred):
    """Computes mean ROC-AUC across targets with class check."""
    aucs = []
    for i in range(y_true.shape[1]):
        # Verify that both positive (1) and negative (0) samples exist in validation split
        if len(np.unique(y_true[:, i])) > 1:
            auc = roc_auc_score(y_true[:, i], y_pred[:, i])
            aucs.append(auc)
            
    return float(np.mean(aucs)) if len(aucs) > 0 else 0.5

# Hyperparameters
EPOCHS = 10
LR = 3e-4
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')  # Automatic Mixed Precision

history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
best_val_auc = 0.0

## 6. Training & Validation Loop

In [ ]:
print("🚀 Starting Training Loop on Fold 0...\n")

for epoch in range(1, EPOCHS + 1):
    # --- TRAIN PHASE ---
    model.train()
    running_train_loss = 0.0
    
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for x_batch, y_batch in train_bar:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        
        # Mixed Precision Forward Pass
        with torch.amp.autocast('cuda'):
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_train_loss += loss.item() * x_batch.size(0)
        train_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    epoch_train_loss = running_train_loss / len(train_dataset)
    scheduler.step()

    # --- VALIDATION PHASE ---
    model.eval()
    running_val_loss = 0.0
    val_preds, val_targets = [], []
    
    with torch.no_grad():
        for x_batch, y_batch in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]"):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            
            with torch.amp.autocast('cuda'):
                logits = model(x_batch)
                loss = criterion(logits, y_batch)
                
            running_val_loss += loss.item() * x_batch.size(0)
            preds = torch.sigmoid(logits)
            
            val_preds.append(preds.cpu().numpy())
            val_targets.append(y_batch.cpu().numpy())
            
    epoch_val_loss = running_val_loss / len(val_dataset)
    val_preds = np.vstack(val_preds)
    val_targets = np.vstack(val_targets)
    
    epoch_val_auc = compute_macro_auc(val_targets, val_preds)
    
    # Store Metrics
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['val_auc'].append(epoch_val_auc)
    
    print(f"📊 Epoch {epoch:02d}/{EPOCHS:02d} | "
          f"Train Loss: {epoch_train_loss:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} | "
          f"Val Macro ROC-AUC: {epoch_val_auc:.4f}")
    
    # Checkpoint Best Model
    if epoch_val_auc > best_val_auc:
        best_val_auc = epoch_val_auc
        torch.save(model.state_dict(), "best_resnet34_fold0.pth")
        print(f"💾 Saved new best checkpoint! (Val Macro ROC-AUC: {best_val_auc:.4f})")
    print("-" * 65)

## 7. Convergence Plotting

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))

color = 'tab:red'
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss', color=color)
ax1.plot(range(1, EPOCHS + 1), history['train_loss'], label='Train Loss', color='crimson', linestyle='--')
ax1.plot(range(1, EPOCHS + 1), history['val_loss'], label='Val Loss', color='darkred')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Val Macro ROC-AUC', color=color)
ax2.plot(range(1, EPOCHS + 1), history['val_auc'], label='Val ROC-AUC', color='dodgerblue', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)

plt.title("RSNA Knee MRI — 2.5D ResNet34 Baseline Training Curves")
fig.tight_layout()
plt.show()

## 8. Next Improvements & Roadmap
- **Semi-Supervised Pre-Training:** Pre-train on the unannotated studies (`fold == -1`) using the soft probability targets (`pseudo_*`) provided in the metadata CSV before fine-tuning on ground-truth folds.
- **Backbone Experimentation:** Test heavier backbones like EfficientNet-B4 or ConvNeXt.
- **3D Data Augmentation:** Introduce spatial transforms such as random horizontal flipping, slight rotations, and intensity gamma shifts via `albumentations`.